# Synthetic Bootstrapping: Distillation and Data Enrichment

Two complementary patterns for generating training data when you don't have any. First, use a capable model to generate a batch of fact-checking examples in a single call. Then walk through data enrichment, where you start from real outputs (jokes) and reverse-engineer the inputs (topics) that a `TellJoke` program would have received.

**Required env vars:** `OPENAI_API_KEY` (loaded from `.env`).

In [ ]:
%pip install -r ../requirements.txt -q

In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM('openai/gpt-5-mini'))

## Model distillation: generate fact-checking examples

A single `Predict` call with `n=15` returns 15 (fact, veracity) pairs we can drop straight into a training set. The seed string nudges the model to vary outputs across runs.

In [ ]:
from typing import List

# Configure a stronger LM for the teacher pass
llm = dspy.LM(model='openai/gpt-5', temperature=1, max_tokens=16000)
dspy.settings.configure(lm=llm)

class FactGeneration(dspy.Signature):
    """Generate facts and their veracity."""
    sindex: str = dspy.InputField(desc="a random string seed")
    fact: str = dspy.OutputField(desc="a statement about the world")
    veracity: bool = dspy.OutputField(desc="True if fact is correct, False otherwise")

# Generate multiple synthetic examples
fact_generator = dspy.Predict(FactGeneration, n=15)
response = fact_generator(sindex="seed_001")

# Convert to training examples
synthetic_examples: List[dspy.Example] = [
    dspy.Example(
        fact=fact,
        answer=veracity,
    ).with_inputs('fact')
    for fact, veracity in zip(
        response.completions.fact,
        response.completions.veracity,
    )
]

print(f"Generated {len(synthetic_examples)} fact-checking examples")
for ex in synthetic_examples[:3]:
    print(ex.fact, '->', ex.answer)

## Data enrichment: reverse-engineer the inputs

Suppose you have a list of jokes you want to train a comedian agent on, but your task signature needs `topic` as the input. Build a second signature that goes from `joke` back to `topic`, then run it over the joke corpus to fill in the missing field.

In [ ]:
# Signature for telling a joke given a topic
class TellJoke(dspy.Signature):
    """Generate a joke given a topic."""
    topic: str = dspy.InputField()
    joke: str = dspy.OutputField()

# Signature for getting a topic from a joke
class IdentifyTopic(dspy.Signature):
    """Generate a topic given a joke."""
    joke: str = dspy.InputField()
    topic: str = dspy.OutputField()

In [ ]:
# A small corpus of jokes you already have on hand
jokes = [
    "Why don't scientists trust atoms? Because they make up everything.",
    "I told my computer I needed a break, and now it won't stop sending me Kit Kats.",
    "Parallel lines have so much in common. It's a shame they'll never meet.",
]

topic_inferrer = dspy.Predict(IdentifyTopic)

enriched_examples = []
for joke in jokes:
    topic = topic_inferrer(joke=joke).topic
    enriched_examples.append(
        dspy.Example(topic=topic, joke=joke).with_inputs('topic')
    )
    print(f"{topic}: {joke}")